# Lesson 2 — Tokens & embeddings (runnable)

Computers don't understand "dog". They understand numbers. We do two things:
**tokenise** (each word → integer id), then **embed** (each id → a list of numbers called a *vector*).

The twist most courses skip: at first the vectors are **random and meaningless**. The magic is that
**training nudges them** until related words end up close. In this notebook we don't just talk about that —
in Step 5 we actually **train tiny embeddings and watch `dog` drift next to `bark`.**

Companion: [`02_tokens_and_embeddings.py`](../02_tokens_and_embeddings.py) · intuition in [`02_walkthrough.md`](../02_walkthrough.md).

## Imports

In [ ]:
import torch                                # PyTorch tensors.
import torch.nn as nn                       # nn.Embedding lives here.
import torch.nn.functional as F             # cross-entropy (used when we TRAIN in Step 5).

torch.manual_seed(0)                        # Reproducible RNG.
torch.set_printoptions(precision=2, sci_mode=False)

## Step 1 — Tokenise

A vocabulary list. Each word's position in the list IS its id number.

In [ ]:
vocab = ["<pad>", "<mask>", "dog", "cat", "fish", "bark", "meow", "swim"]
tok2id = {w: i for i, w in enumerate(vocab)}     # word → id lookup.

print("Vocabulary:")
for i, w in enumerate(vocab):
    print(f"  id={i}  word={w}")

# Encoding a sentence = look up each word's ID
sentence = ["dog", "bark"]
ids = [tok2id[w] for w in sentence]              # Convert each word to id.
print(f"\nSentence {sentence}  ->  ids {ids}")
print(f"SHAPE TRAIL:  {len(sentence)} words  ->  {len(ids)} ids   (one id per word)")

## Step 2 — Embed

The embedding table is just a 2-D grid of numbers. Row `i` holds the vector for word `i`.
Looking up a word = grabbing its row. No math, just a lookup.

In [ ]:
embedding_dim = 4                              # 4 numbers per word.
emb = nn.Embedding(len(vocab), embedding_dim)  # 8×4 trainable lookup table.

print(f"Embedding table shape: {tuple(emb.weight.shape)}   <- {len(vocab)} words × {embedding_dim} numbers each\n")

print("Word -> vector:")
for word in ["dog", "cat", "fish"]:
    v = emb(torch.tensor(tok2id[word])).detach()
    print(f"  {word:5s} ->  {v}")

⚠️ **These vectors are RANDOM right now** — the table was just filled with noise.
`dog`'s numbers mean nothing yet. That's expected! Nobody types in what a word means.

Keep this in mind: in **Step 5** we'll train the table and these same numbers will become meaningful.

## Step 3 — Encoding a whole sentence at once

In [ ]:
ids_tensor = torch.tensor(ids)
vectors = emb(ids_tensor)                       # Look up both rows at once.
print(f"Sentence vectors:  shape {tuple(vectors.shape)}   <- {ids_tensor.shape[0]} words × {embedding_dim} numbers")
print(vectors.detach())
print("\nSHAPE TRAIL:  2 ids  ->  (2 × 4) block of numbers   (each word became a 4-number vector)")

## Step 4 — Similarity via dot product

To ask *"how alike are two words?"* we use the **dot product**: multiply matching slots, add the products.
Big total = similar direction; near zero or negative = unrelated.

Right now the embeddings are random, so these numbers are **meaningless** — notice there's no pattern:

In [ ]:
def similarity(table, a, b):
    va = table(torch.tensor(tok2id[a]))
    vb = table(torch.tensor(tok2id[b]))
    return torch.dot(va, vb).item()

print("Similarity with RANDOM embeddings (should look like noise):")
print(f"  dog · bark  = {similarity(emb, 'dog', 'bark'):+.2f}   (we WISH this were big — they go together)")
print(f"  dog · meow  = {similarity(emb, 'dog', 'meow'):+.2f}   (we WISH this were small — unrelated)")
print(f"  cat · meow  = {similarity(emb, 'cat', 'meow'):+.2f}")
print("\nNo pattern. Random numbers in, random similarity out. Let's fix that by TRAINING.")

## Step 5 — Watch the embeddings actually LEARN 🌱

Here's the part every other explanation skips. We'll give the model a tiny game and let the **5-line training loop**
reshape the embeddings.

**The game (a baby version of how real word2vec/BERT learn):** *given a word, guess its partner.*
Our secret pairs are `dog↔bark`, `cat↔meow`, `fish↔swim`. To win, the model must make each word's vector
**point toward its partner's vector** (so their dot product is the biggest). Nobody tells it the coordinates —
it discovers them by playing.

We use `embedding_dim = 2` this time so we can **draw the result on a 2-D map** at the end.

In [ ]:
pairs = [("dog", "bark"), ("cat", "meow"), ("fish", "swim")]
train_pairs = pairs + [(b, a) for a, b in pairs]    # learn both directions: dog->bark AND bark->dog

torch.manual_seed(1)                                # fresh start
emb2 = nn.Embedding(len(vocab), 2)                  # 2 numbers per word so we can PLOT it
opt  = torch.optim.Adam(emb2.parameters(), lr=0.1)  # the optimiser from L1

# snapshot the "before" similarities (random)
watch = [("dog","bark"), ("dog","meow"), ("cat","meow"), ("cat","swim")]
before = {p: similarity(emb2, *p) for p in watch}

for step in range(400):                             # the SAME loop you saw in L1, 400 times
    loss = 0.0
    for a, b in train_pairs:                        # each example: word a should predict partner b
        # dot a's vector with EVERY word -> a score per word (V scores)
        logits = emb2.weight @ emb2(torch.tensor(tok2id[a]))
        loss = loss + F.cross_entropy(logits.unsqueeze(0), torch.tensor([tok2id[b]]))
    opt.zero_grad(); loss.backward(); opt.step()    # clear, backprop, nudge the vectors
    if step % 100 == 0:
        print(f"step {step:3d}   loss {loss.item():.3f}")

after = {p: similarity(emb2, *p) for p in watch}
print("\nDone. The embeddings have moved. Let's compare before vs after. ⬇")

### Before → after: did `dog` and `bark` get closer?

In [ ]:
print(f"{'pair':<14}{'BEFORE':>9}{'AFTER':>9}    meaning")
print("-" * 48)
for p in watch:
    a, b = p
    relation = "partners ✅ (want BIG)" if (a, b) in train_pairs else "unrelated ❌ (want small)"
    arrow = "  ↑" if after[p] > before[p] else "  ↓"
    print(f"{a+' · '+b:<14}{before[p]:>+9.2f}{after[p]:>+9.2f}{arrow}  {relation}")
print("\nThe partner pairs shot UP; the unrelated pairs got pushed DOWN.")
print("Nobody programmed that — the training loop discovered it from the game.")

### Draw the map 🗺

Each word now lives at a 2-D point. The dot product cares most about **direction** (which way a vector points),

so we shrink every vector to length 1 and plot it — now all 6 words sit around a circle and you can see

**each animal landing near its sound.**

In [ ]:
import math

content = ["dog", "bark", "cat", "meow", "fish", "swim"]
unit = {}
for w in content:
    v = emb2(torch.tensor(tok2id[w])).detach()
    v = v / (v.norm() + 1e-9)               # length 1: only DIRECTION matters (that's what the dot product sees)
    unit[w] = tuple(v.tolist())

# (a) The clincher: each word's compass angle. Sort it — partners should sit next to each other.
def angle(v): return math.degrees(math.atan2(v[1], v[0])) % 360
rows = sorted(content, key=lambda w: angle(unit[w]))
print("Each word as a compass direction (sorted by angle):\n")
print(f"  {'word':<6}{'angle':>7}")
print("  " + "-" * 15)
prev = None
for w in rows:
    a = angle(unit[w])
    near = "  <- almost same angle as the one above!" if prev is not None and abs(a - prev) < 25 else ""
    print(f"  {w:<6}{a:>6.0f}°{near}")
    prev = a
print("\nPartners land at nearly the SAME angle = they point the same way = big dot product. 🎯")

# (b) The map. Partners overlap (that's the point!), so nudge labels to free rows so both show.
def ascii_map(points, width=54, height=21):
    xs = [p[0] for p in points.values()]; ys = [p[1] for p in points.values()]
    xmin, xmax = min(xs), max(xs); ymin, ymax = min(ys), max(ys)
    px = (xmax - xmin or 1) * 0.25; py = (ymax - ymin or 1) * 0.25
    xmin -= px; xmax += px; ymin -= py; ymax += py
    grid = [[" "] * width for _ in range(height)]
    if xmin < 0 < xmax:
        c0 = int((0 - xmin) / (xmax - xmin) * (width - 1))
        for r in range(height): grid[r][c0] = "."
    if ymin < 0 < ymax:
        r0 = height - 1 - int((0 - ymin) / (ymax - ymin) * (height - 1))
        for c in range(width): grid[r0][c] = "."
    def free(row, col, n):
        if not (0 <= row < height): return False
        return all(0 <= col + k < width and grid[row][col + k] == " " for k in range(n))
    for w, (x, y) in points.items():
        col = int((x - xmin) / (xmax - xmin) * (width - 1))
        row = height - 1 - int((y - ymin) / (ymax - ymin) * (height - 1))
        label = "*" + w
        r = row
        for off in [0, -1, 1, -2, 2]:        # if a partner already sits here, drop to a nearby row
            if free(row + off, col, len(label)): r = row + off; break
        for k, ch in enumerate(label):
            c = col + k
            if 0 <= r < height and 0 <= c < width: grid[r][c] = ch
    print("\n".join("".join(rw) for rw in grid))

print("\nMap of trained word DIRECTIONS (. = the x=0 / y=0 lines):\n")
ascii_map(unit)
print("\nEach animal sits right on top of (or touching) its partner sound — same direction!")
print("Re-run Step 5 with a different seed: the whole map rotates, but partners stay glued together.")

## Things to try

1. In Step 5, change `range(400)` to `range(20)`. The map is still messy — it needs more training.
2. Add a new pair, e.g. `("bird", "tweet")` — but first add those words to `vocab`. Do they cluster too?
3. Bump `embedding_dim` back to 4 in Step 5 (you lose the 2-D map, but print the before/after table — does it still work?).
4. (Stretch) After training, loop over every word and print the OTHER word with the highest dot product. Are they always partners?